# Parte 2: Del LLM al clasificador
**Orden de trabajo:** el split se hace primero (aunque en el enunciado es el paso 3) para que el zero-shot, el LLM y el Transformer se evalúen sobre el mismo conjunto de test, que no se toca hasta la evaluación final.

## 1. Partición de datos 70/15/15 (paso 3 del enunciado)
Split estratificado por `etiqueta` con `random_state=42`.

In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

df = pd.read_csv("datos/noticias.csv")
LABELS = ["politica", "economia", "deportes", "ciencia_tecnologia"]

train, temp = train_test_split(df, test_size=0.30, stratify=df["etiqueta"], random_state=42)
val, test = train_test_split(temp, test_size=0.50, stratify=temp["etiqueta"], random_state=42)
print(len(train), len(val), len(test))

280 60 60


### 1.1 Noticias por categoría en cada split

In [2]:
pd.DataFrame({"train": train["etiqueta"].value_counts(),
              "val": val["etiqueta"].value_counts(),
              "test": test["etiqueta"].value_counts()})

,train,val,test
etiqueta,,,
ciencia_tecnologia,70,15,15
deportes,70,15,15
economia,70,15,15
politica,70,15,15


## 2. Línea base zero-shot (paso 1 del enunciado)
40 titulares del test clasificados con `Recognai/bert-base-spanish-wwm-cased-xnli`, sin fine-tuning. Este F1 macro es el piso mínimo a superar.

In [3]:
from transformers import pipeline

zs = pipeline("zero-shot-classification", model="Recognai/bert-base-spanish-wwm-cased-xnli")
candidatas = {"política": "politica", "economía": "economia",
              "deportes": "deportes", "ciencia y tecnología": "ciencia_tecnologia"}

muestra = test.sample(40, random_state=42)
salidas = zs(muestra["titular"].tolist(), candidate_labels=list(candidatas),
             hypothesis_template="Esta noticia trata sobre {}.")
muestra["pred_zs"] = [candidatas[o["labels"][0]] for o in salidas]

acc_zs = accuracy_score(muestra["etiqueta"], muestra["pred_zs"])
f1_zs = f1_score(muestra["etiqueta"], muestra["pred_zs"], average="macro")
print(f"Zero-shot -> Accuracy: {acc_zs:.3f} | F1 macro: {f1_zs:.3f}")

/opt/anaconda3/envs/tarea1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1018 23:26:23.981000 10892 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 13694.63it/s]


Zero-shot -> Accuracy: 0.575 | F1 macro: 0.571
